# E-Commerce Price Prediction - Model Testing

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle as pkl

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

In [2]:
train_df = pd.read_csv('./ecommerce_price_prediction-train.csv')
test_df = pd.read_csv('./ecommerce_price_prediction-test-3-days.csv')
preprocessed_train_df = pd.read_csv('./ecommerce_price_prediction-train-preprocessed.csv')

## Data Preprocessing

In [3]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25900 entries, 0 to 25899
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   capturedAt                25900 non-null  object 
 1   shopId                    25900 non-null  int64  
 2   itemId                    25900 non-null  int64  
 3   modelId                   25900 non-null  int64  
 4   price                     300 non-null    float64
 5   priceBeforeDiscount       300 non-null    float64
 6   promotionId               300 non-null    float64
 7   cat_id                    300 non-null    float64
 8   stock                     7 non-null      float64
 9   normal_stock              7 non-null      float64
 10  raw_discount              300 non-null    float64
 11  show_discount             300 non-null    float64
 12  brand                     166 non-null    object 
 13  is_free_shipping          300 non-null    object 
 14  is_pre

In [4]:
test_df.describe()

,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,raw_discount,show_discount,item_price_min,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count
count,2.590000e+04,2.590000e+04,2.590000e+04,3.000000e+02,3.000000e+02,3.000000e+02,300.000000,7.0,7.0,300.000000,300.000000,3.000000e+02,3.000000e+02,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000
mean,7.087618e+08,2.059444e+10,1.746367e+11,3.413033e+07,1.779333e+07,2.232564e+14,100399.940000,0.0,0.0,22.820000,22.820000,3.066700e+07,4.090733e+07,4.711777,126.300000,119.653333,4.943701,76.860000,7773.026667
std,4.464565e+08,7.807730e+09,6.700510e+10,1.030374e+08,3.658597e+07,2.764533e+14,300.095698,0.0,0.0,29.471163,29.471163,1.028034e+08,1.098917e+08,1.084748,387.568956,379.170270,0.044304,17.976026,22700.086828
min,1.003777e+06,6.632660e+07,5.246531e+07,3.000000e+05,0.000000e+00,0.000000e+00,100001.000000,0.0,0.0,0.000000,0.000000,3.000000e+05,5.000000e+05,0.000000,0.000000,0.000000,4.666306,7.000000,1.000000
25%,1.005251e+08,1.959431e+10,1.389479e+11,6.900000e+06,0.000000e+00,0.000000e+00,100015.000000,0.0,0.0,0.000000,0.000000,5.900000e+06,9.275000e+06,4.937557,5.000000,5.000000,4.938943,59.000000,158.000000
50%,1.005099e+09,2.287777e+10,1.876642e+11,1.370000e+07,0.000000e+00,0.000000e+00,100630.000000,0.0,0.0,0.000000,0.000000,1.100000e+07,1.670000e+07,4.980178,20.500000,19.000000,4.955628,82.000000,858.000000
75%,1.007169e+09,2.529063e+10,2.254733e+11,3.252500e+07,2.477500e+07,5.475751e+14,100636.000000,0.0,0.0,50.000000,50.000000,2.650000e+07,4.230000e+07,5.000000,99.000000,94.000000,4.973496,93.000000,3939.000000
max,1.010119e+09,2.996089e+10,2.578670e+11,1.660000e+09,3.250000e+08,6.124352e+14,100640.000000,0.0,0.0,80.000000,80.000000,1.660000e+09,1.660000e+09,5.000000,4829.000000,4827.000000,5.000000,100.000000,101744.000000


In [5]:
test_df = test_df.drop(['stock','normal_stock'], axis = 1)
test_df

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,raw_discount,show_discount,...,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25895,2025-03-24 22:14:56.834,10058821,531851278,417717331,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25896,2025-03-24 22:18:38.339,10058821,2777077096,61026396385,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25897,2025-03-24 22:48:31.578,1009293775,25056411106,197251937839,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25898,2025-03-24 23:17:18.217,10055250,19262129512,213179841270,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
onehot_cols = ["is_free_shipping", "is_pre_order", "is_official_shop", "is_verified", "is_preferred_plus_seller"]

for col in onehot_cols:
    test_df[col] = test_df[col].apply(lambda x : 1 if x == 't' else 0)

In [7]:
test_df['raw_discount'] = np.where(test_df['raw_discount'] == 0, np.nan, test_df['raw_discount'])
test_df['promotionId'] = np.where(test_df['promotionId'] == 0, np.nan, test_df['promotionId'])

test_df

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,raw_discount,show_discount,...,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25895,2025-03-24 22:14:56.834,10058821,531851278,417717331,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25896,2025-03-24 22:18:38.339,10058821,2777077096,61026396385,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25897,2025-03-24 22:48:31.578,1009293775,25056411106,197251937839,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25898,2025-03-24 23:17:18.217,10055250,19262129512,213179841270,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [8]:
test_df['priceBeforeDiscount'] = np.where(test_df['priceBeforeDiscount'] == 0, test_df['price'], test_df['priceBeforeDiscount'])

test_df

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,raw_discount,show_discount,...,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25895,2025-03-24 22:14:56.834,10058821,531851278,417717331,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25896,2025-03-24 22:18:38.339,10058821,2777077096,61026396385,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25897,2025-03-24 22:48:31.578,1009293775,25056411106,197251937839,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25898,2025-03-24 23:17:18.217,10055250,19262129512,213179841270,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [9]:
test_df = test_df.drop(['show_discount'], axis = 1)
test_df

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,raw_discount,brand,...,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25895,2025-03-24 22:14:56.834,10058821,531851278,417717331,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25896,2025-03-24 22:18:38.339,10058821,2777077096,61026396385,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25897,2025-03-24 22:48:31.578,1009293775,25056411106,197251937839,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25898,2025-03-24 23:17:18.217,10055250,19262129512,213179841270,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [10]:
# load encoder used in training phase and use it for test dataset

with open('brand_encoder.pkl', 'rb') as file:
    enc = pkl.load(file)
    
test_df['brand'] = enc.transform(test_df[['brand']])
test_df

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,raw_discount,brand,...,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25895,2025-03-24 22:14:56.834,10058821,531851278,417717331,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25896,2025-03-24 22:18:38.339,10058821,2777077096,61026396385,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25897,2025-03-24 22:48:31.578,1009293775,25056411106,197251937839,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
25898,2025-03-24 23:17:18.217,10055250,19262129512,213179841270,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [11]:
for col in test_df.columns:
    if col=='capturedAt' or col=='shopId' or col=='itemId' or col=='modelId' or col=='price' :
        continue
    col_name = str(col) + "_is_missing"
    test_df[col_name] = test_df[col].isna().astype(int)
    
test_df

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,raw_discount,brand,...,item_price_max_is_missing,review_rating_is_missing,total_rating_count_is_missing,cmt_count_is_missing,shop_rating_is_missing,shop_response_rate_is_missing,shop_follower_count_is_missing,is_official_shop_is_missing,is_verified_is_missing,is_preferred_plus_seller_is_missing
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,0,0,0
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,0,0,0
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,0,0,0
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,0,0,0
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25895,2025-03-24 22:14:56.834,10058821,531851278,417717331,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,0,0,0
25896,2025-03-24 22:18:38.339,10058821,2777077096,61026396385,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,0,0,0
25897,2025-03-24 22:48:31.578,1009293775,25056411106,197251937839,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,0,0,0
25898,2025-03-24 23:17:18.217,10055250,19262129512,213179841270,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,1,0,0,0


In [12]:
test_df['capturedAt'] = pd.to_datetime(test_df['capturedAt'])
test_df['day_of_week'] = test_df['capturedAt'].dt.dayofweek
test_df['hour'] = test_df['capturedAt'].dt.hour
test_df['minute'] = test_df['capturedAt'].dt.minute
test_df['is_holiday'] = np.where(test_df['day_of_week'] >= 5,1,0)
test_df['capturedAt'] = test_df['capturedAt'].astype('int64')

test_df

C:\Users\Greg\AppData\Local\Temp/ipykernel_19532/823957205.py:6: FutureWarning: casting datetime64[ns] values to int64 with .astype(...) is deprecated and will raise in a future version. Use .view(...) instead.
  test_df['capturedAt'] = test_df['capturedAt'].astype('int64')


,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,raw_discount,brand,...,shop_rating_is_missing,shop_response_rate_is_missing,shop_follower_count_is_missing,is_official_shop_is_missing,is_verified_is_missing,is_preferred_plus_seller_is_missing,day_of_week,hour,minute,is_holiday
0,1742617625325000000,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,5,4,27,1
1,1742617625325000000,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,5,4,27,1
2,1742617625325000000,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,5,4,27,1
3,1742617625325000000,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,5,4,27,1
4,1742617625325000000,1009757562,27904817781,241162534604,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,5,4,27,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25895,1742854496834000000,10058821,531851278,417717331,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,0,22,14,0
25896,1742854718339000000,10058821,2777077096,61026396385,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,0,22,18,0
25897,1742856511578000000,1009293775,25056411106,197251937839,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,0,22,48,0
25898,1742858238217000000,10055250,19262129512,213179841270,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,0,23,17,0


In [13]:
test_df['has_data'] = test_df['price'].isna().astype(int)
test_df = test_df.sort_values(by=['has_data','capturedAt'], ascending=[True,True])
test_df = test_df.drop(['has_data'],axis = 1)

test_df

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,raw_discount,brand,...,shop_rating_is_missing,shop_response_rate_is_missing,shop_follower_count_is_missing,is_official_shop_is_missing,is_verified_is_missing,is_preferred_plus_seller_is_missing,day_of_week,hour,minute,is_holiday
50,1742617625388000000,1009757562,22983470729,241402913254,42500000.0,42500000.0,NaN,100001.0,NaN,288.0,...,0,0,0,0,0,0,5,4,27,1
131,1742619212412000000,1010118847,22760898836,255025844230,28500000.0,28500000.0,NaN,100636.0,NaN,NaN,...,0,0,0,0,0,0,5,4,53,1
315,1742619212839000000,1007168904,25014396500,214922892911,11500000.0,18900000.0,6.077567e+14,100636.0,39.0,0.0,...,0,0,0,0,0,0,5,4,53,1
321,1742619212839000000,1007168904,25014396500,214922892911,11500000.0,18900000.0,6.077567e+14,100636.0,39.0,0.0,...,0,0,0,0,0,0,5,4,53,1
416,1742619213118000000,1005890827,24233726195,187664166805,6900000.0,6900000.0,NaN,100532.0,NaN,NaN,...,0,0,0,0,0,0,5,4,53,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25894,1742854496834000000,10058821,531851278,417717329,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,0,22,14,0
25895,1742854496834000000,10058821,531851278,417717331,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,0,22,14,0
25896,1742854718339000000,10058821,2777077096,61026396385,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,0,22,18,0
25897,1742856511578000000,1009293775,25056411106,197251937839,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,1,0,0,0,0,22,48,0


In [14]:
item_stats = train_df.groupby('itemId')['price'].agg(['mean','std'])
item_stats.columns = ['item_price_mean', 'item_price_std']

shop_stats = train_df.groupby('shopId')['price'].agg(['mean','std'])
shop_stats.columns = ['shop_price_mean', 'shop_price_std']

shop_disc_stats = train_df.groupby('shopId')['raw_discount'].agg(['mean','count'])
shop_disc_stats.columns = ['shop_discount_mean','shop_discount_count']

test_df = test_df.merge(item_stats, on='itemId', how='left')
test_df = test_df.merge(shop_stats, on='shopId', how='left')
test_df = test_df.merge(shop_disc_stats, on='shopId', how='left')

test_df

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,raw_discount,brand,...,day_of_week,hour,minute,is_holiday,item_price_mean,item_price_std,shop_price_mean,shop_price_std,shop_discount_mean,shop_discount_count
0,1742617625388000000,1009757562,22983470729,241402913254,42500000.0,42500000.0,NaN,100001.0,NaN,288.0,...,5,4,27,1,2.620455e+07,1.076101e+07,2.257315e+07,2.760786e+07,0.170645,20024
1,1742619212412000000,1010118847,22760898836,255025844230,28500000.0,28500000.0,NaN,100636.0,NaN,NaN,...,5,4,53,1,2.916667e+07,4.715991e+05,1.768876e+07,1.315354e+07,0.000000,2136
2,1742619212839000000,1007168904,25014396500,214922892911,11500000.0,18900000.0,6.077567e+14,100636.0,39.0,0.0,...,5,4,53,1,1.150000e+07,0.000000e+00,2.015301e+07,3.097746e+07,36.765286,21392
3,1742619212839000000,1007168904,25014396500,214922892911,11500000.0,18900000.0,6.077567e+14,100636.0,39.0,0.0,...,5,4,53,1,1.150000e+07,0.000000e+00,2.015301e+07,3.097746e+07,36.765286,21392
4,1742619213118000000,1005890827,24233726195,187664166805,6900000.0,6900000.0,NaN,100532.0,NaN,NaN,...,5,4,53,1,6.900000e+06,0.000000e+00,6.900000e+06,0.000000e+00,0.000000,870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25895,1742854496834000000,10058821,531851278,417717329,NaN,NaN,NaN,NaN,NaN,NaN,...,0,22,14,0,8.500000e+07,0.000000e+00,8.506025e+07,4.761537e+07,0.000000,11327
25896,1742854496834000000,10058821,531851278,417717331,NaN,NaN,NaN,NaN,NaN,NaN,...,0,22,14,0,8.500000e+07,0.000000e+00,8.506025e+07,4.761537e+07,0.000000,11327
25897,1742854718339000000,10058821,2777077096,61026396385,NaN,NaN,NaN,NaN,NaN,NaN,...,0,22,18,0,4.950000e+07,0.000000e+00,8.506025e+07,4.761537e+07,0.000000,11327
25898,1742856511578000000,1009293775,25056411106,197251937839,NaN,NaN,NaN,NaN,NaN,NaN,...,0,22,48,0,2.150000e+07,0.000000e+00,1.169827e+07,4.412088e+06,0.000000,1791


In [15]:
# function to implement recalibration using anchor set data
def anchor_set_recalibration(model, day, num_data):
    for i in range(day) :
        base_anchor = test_df[i*num_data:(i+1)*num_data]
        train_sample = preprocessed_train_df.sample(n=10000,random_state=42)
        anchor_set = pd.concat((base_anchor, train_sample), axis = 0)
        X_anchor = anchor_set.drop(['price'], axis = 1)
        y_anchor = anchor_set['price']
        model.fit(X_anchor, y_anchor)
        
# function to run live prediction on test dataset
def predict_in_day(model, day):
    global result_df
    test_now = test_df[(pd.to_datetime(test_df['capturedAt'])).dt.normalize() == day]
    X_test_now = test_now.drop(['price'], axis = 1)
    y_pred_now = model.predict(X_test_now)
    test_index = test_now.index
    for i in range(len(test_index)):
        result_df.loc[test_index[i],'predicted_price'] = y_pred_now[i]

In [16]:
# model testing

result_df = test_df.copy()
result_df['predicted_price'] = result_df['price']

days = 3
num_data = 100
with open('best_model.pkl','rb') as file:
    model = pkl.load(file)

for day in range(days) :
    anchor_set_recalibration(model,day,num_data)
    current_day = pd.to_datetime(test_df.loc[day*num_data,'capturedAt']).normalize()
#     print(day, day*num_data, current_day)
    predict_in_day(model,current_day)

In [17]:
# check results with anchor set data

anchor_index = result_df[result_df['price'].isna() == False].index
y_true = result_df.loc[anchor_index,'price']
y_pred = result_df.loc[anchor_index,'predicted_price']
print("RMSE = " + str(root_mean_squared_error(y_true, y_pred)))
print("MAE = " + str(mean_absolute_error(y_true, y_pred)))
print("MAPE = " + str(mean_absolute_percentage_error(y_true, y_pred)))
print("R-squared = " + str(r2_score(y_true, y_pred)))

RMSE = 4518455.4478111
MAE = 1114715.8963253996
MAPE = 0.07021738271035662
R-squared = 0.998070519925278


In [18]:
# save results to new csv file

result_df = result_df.sort_index()
result_df.to_csv('ecommerce_price_prediction-results.csv')